In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded")

✅ Бібліотеки завантажено


## Step 1: Loading Data 2012-2025

In [ ]:
data_path = Path('../data/tml')

START_YEAR = 2012
END_YEAR = 2025

all_matches = []

print("Loading files:")
print("=" * 60)

for year in range(START_YEAR, END_YEAR + 1):
    file_path = data_path / f'{year}.csv'
    
    if file_path.exists():
        df_year = pd.read_csv(file_path)
        df_year['data_year'] = year
        all_matches.append(df_year)
        print(f"  {year}: {len(df_year):,} matches | {len(df_year.columns)} columns")
    else:
        print(f"  {year}: file not found")

print("=" * 60)

df_all = pd.concat(all_matches, ignore_index=True)

print(f"\nDONE!")
print(f"Total matches: {len(df_all):,}")
print(f"Columns: {len(df_all.columns)}")
print(f"Period: {START_YEAR}-{END_YEAR}")
print(f"Memory usage: {df_all.memory_usage(deep=True).sum() / (1024**2):.2f} MB")

Завантаження файлів:
✓ 2012: 3,009 матчів | 51 колонок
✓ 2013: 2,944 матчів | 51 колонок
✓ 2014: 2,901 матчів | 51 колонок
✓ 2015: 2,945 матчів | 51 колонок
✓ 2016: 2,970 матчів | 51 колонок
✓ 2017: 2,936 матчів | 51 колонок
✓ 2018: 2,926 матчів | 51 колонок
✓ 2019: 2,806 матчів | 51 колонок
✓ 2020: 1,466 матчів | 51 колонок
✓ 2021: 2,736 матчів | 51 колонок
✓ 2022: 2,918 матчів | 51 колонок
✓ 2023: 2,995 матчів | 51 колонок
✓ 2024: 3,076 матчів | 51 колонок
✓ 2025: 2,915 матчів | 51 колонок

✅ ГОТОВО!
Загальна кількість матчів: 39,543
Колонок: 51
Період: 2012-2025
Розмір у пам'яті: 48.44 MB


## Step 2: Data Quality Check

In [ ]:
print("DATA STRUCTURE")
print("=" * 80)
print(f"Shape: {df_all.shape}")
print(f"\nFirst 5 columns: {df_all.columns[:5].tolist()}")
print(f"Last 5 columns: {df_all.columns[-5:].tolist()}")
print(f"\nData types:")
print(df_all.dtypes.value_counts())

print("\n" + "=" * 80)
print("DISTRIBUTION BY YEAR")
print("=" * 80)
year_dist = df_all['data_year'].value_counts().sort_index()
for year, count in year_dist.items():
    print(f"{year}: {count:,} matches")

print("\n" + "=" * 80)
print("DATA SAMPLE")
print("=" * 80)
print("\nFirst 3 rows:")
print(df_all.head(3))

📊 СТРУКТУРА ДАНИХ
Форма датафрейму: (39543, 51)

Перші 5 колонок: ['tourney_id', 'tourney_name', 'surface', 'draw_size', 'tourney_level']
Останні 5 колонок: ['l_2ndWon', 'l_SvGms', 'l_bpSaved', 'l_bpFaced', 'data_year']

Типи даних:
float64    32
object     17
int64       2
Name: count, dtype: int64

📅 РОЗПОДІЛ ПО РОКАХ
2012: 3,009 матчів
2013: 2,944 матчів
2014: 2,901 матчів
2015: 2,945 матчів
2016: 2,970 матчів
2017: 2,936 матчів
2018: 2,926 матчів
2019: 2,806 матчів
2020: 1,466 матчів
2021: 2,736 матчів
2022: 2,918 матчів
2023: 2,995 матчів
2024: 3,076 матчів
2025: 2,915 матчів

🔎 ПРИКЛАД ДАНИХ

Перші 3 рядки:
  tourney_id tourney_name surface  draw_size tourney_level indoor  \
0   2012-339     Brisbane    Hard       32.0           250      O   
1   2012-339     Brisbane    Hard       32.0           250      O   
2   2012-339     Brisbane    Hard       32.0           250      O   

   tourney_date  match_num winner_id  winner_seed  ... l_ace l_df l_svpt  \
0      20120101        1.0

In [92]:
import pandas as pd       # для роботи з таблицями (DataFrame)
import numpy as np  
import matplotlib.pyplot as plt   # графіки
import seaborn as sns 
from sklearn.model_selection import train_test_split   # розбиття на train/test
from sklearn.metrics import accuracy_score, classification_report, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler      # нормалізація / масштабування фіч
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor

## Step 3: Missing Values Analysis

In [ ]:
missing_stats = pd.DataFrame({
    'Missing_Count': df_all.isnull().sum(),
    'Missing_Percent': (df_all.isnull().sum() / len(df_all) * 100).round(2)
})

missing_stats = missing_stats[missing_stats['Missing_Count'] > 0].sort_values(
    'Missing_Percent', ascending=False
)

⚠️  ПРОПУЩЕНІ ЗНАЧЕННЯ (топ-20)
              Кількість_NaN  Відсоток_NaN
winner_entry          34386         86.96
loser_entry           31402         79.41
loser_seed            30033         75.95
winner_seed           22846         57.78
indoor                 3003          7.59
minutes                2980          7.54
l_bpFaced              2352          5.95
l_df                   2352          5.95
l_ace                  2352          5.95
w_bpFaced              2352          5.95
w_df                   2352          5.95
w_ace                  2352          5.95
l_1stIn                2352          5.95
w_svpt                 2352          5.95
w_1stIn                2352          5.95
w_1stWon               2352          5.95
w_2ndWon               2352          5.95
l_svpt                 2352          5.95
w_SvGms                2352          5.95
w_bpSaved              2352          5.95

📈 Всього колонок з пропусками: 44
📊 Всього колонок: 51
✅ Колонок без пропусків: 7


## Step 4: Handling Missing Values

Strategy:
- **hand, ht, age, ioc** (player metadata): fill or remove
- **winner_seed, loser_seed** (seeds): keep NaN (means "unseeded")
- **winner_entry, loser_entry** (entry type): keep NaN or fill with category
- **indoor** (indoor/outdoor): keep NaN (unknown)
- **minutes** (duration): keep NaN (not critical)
- **Match statistics** (w_ace, l_df, etc.): **keep NaN** - will use this data to create rolling features from past matches

In [ ]:
print("HANDLING MISSING VALUES")
print("=" * 80)

initial_size = len(df_all)
print(f"Initial size: {initial_size:,} matches\n")

# 1. Player metadata
print("1. PLAYER METADATA")
print("-" * 80)

if 'winner_hand' in df_all.columns:
    before = df_all['winner_hand'].isna().sum()
    df_all['winner_hand'] = df_all['winner_hand'].fillna('R')
    print(f"  winner_hand: filled {before:,} missing → 'R'")

if 'loser_hand' in df_all.columns:
    before = df_all['loser_hand'].isna().sum()
    df_all['loser_hand'] = df_all['loser_hand'].fillna('R')
    print(f"  loser_hand: filled {before:,} missing → 'R'")

if 'winner_ht' in df_all.columns:
    before = df_all['winner_ht'].isna().sum()
    median_ht = df_all['winner_ht'].median()
    df_all['winner_ht'] = df_all['winner_ht'].fillna(median_ht)
    print(f"  winner_ht: filled {before:,} missing → {median_ht:.0f} cm")

if 'loser_ht' in df_all.columns:
    before = df_all['loser_ht'].isna().sum()
    median_ht = df_all['loser_ht'].median()
    df_all['loser_ht'] = df_all['loser_ht'].fillna(median_ht)
    print(f"  loser_ht: filled {before:,} missing → {median_ht:.0f} cm")

if 'winner_age' in df_all.columns:
    before = df_all['winner_age'].isna().sum()
    median_age = df_all['winner_age'].median()
    df_all['winner_age'] = df_all['winner_age'].fillna(median_age)
    print(f"  winner_age: filled {before:,} missing → {median_age:.1f} years")

if 'loser_age' in df_all.columns:
    before = df_all['loser_age'].isna().sum()
    median_age = df_all['loser_age'].median()
    df_all['loser_age'] = df_all['loser_age'].fillna(median_age)
    print(f"  loser_age: filled {before:,} missing → {median_age:.1f} years")

ioc_cols = [col for col in ['winner_ioc', 'loser_ioc'] if col in df_all.columns]
if ioc_cols:
    before = len(df_all)
    df_all = df_all.dropna(subset=ioc_cols)
    dropped = before - len(df_all)
    print(f"  ioc: dropped {dropped:,} rows with missing values")

# 2. Entry type
print("\n2. ENTRY TYPE")
print("-" * 80)

if 'winner_entry' in df_all.columns:
    before = df_all['winner_entry'].isna().sum()
    df_all['winner_entry'] = df_all['winner_entry'].fillna('DA')
    print(f"  winner_entry: filled {before:,} missing → 'DA' (Direct Acceptance)")

if 'loser_entry' in df_all.columns:
    before = df_all['loser_entry'].isna().sum()
    df_all['loser_entry'] = df_all['loser_entry'].fillna('DA')
    print(f"  loser_entry: filled {before:,} missing → 'DA' (Direct Acceptance)")

# 3. Rank and rank points
print("\n3. RANK AND RANK_POINTS")
print("-" * 80)

if 'winner_rank' in df_all.columns:
    before = df_all['winner_rank'].isna().sum()
    df_all['winner_rank'] = df_all['winner_rank'].fillna(1000)
    print(f"  winner_rank: filled {before:,} → 1000 (weak player)")

if 'loser_rank' in df_all.columns:
    before = df_all['loser_rank'].isna().sum()
    df_all['loser_rank'] = df_all['loser_rank'].fillna(1000)
    print(f"  loser_rank: filled {before:,} → 1000")

if 'winner_rank_points' in df_all.columns:
    before = df_all['winner_rank_points'].isna().sum()
    df_all['winner_rank_points'] = df_all['winner_rank_points'].fillna(0)
    print(f"  winner_rank_points: filled {before:,} → 0")

if 'loser_rank_points' in df_all.columns:
    before = df_all['loser_rank_points'].isna().sum()
    df_all['loser_rank_points'] = df_all['loser_rank_points'].fillna(0)
    print(f"  loser_rank_points: filled {before:,} → 0")

# 4. Draw size
print("\n4. DRAW_SIZE")
print("-" * 80)

if 'draw_size' in df_all.columns:
    before = df_all['draw_size'].isna().sum()
    if before > 0:
        mode_value = df_all['draw_size'].mode()[0] if len(df_all['draw_size'].mode()) > 0 else 32
        df_all['draw_size'] = df_all['draw_size'].fillna(mode_value)
        print(f"  draw_size: filled {before:,} → {int(mode_value)} (most common)")
    else:
        print(f"  draw_size: no missing values")

# 5. Match num
print("\n5. MATCH_NUM")
print("-" * 80)

if 'match_num' in df_all.columns:
    df_all = df_all.drop(columns=['match_num'])
    print(f"  match_num: removed (not needed for model)")
else:
    print(f"  match_num: column not present")

# 6. Seed
print("\n6. SEED")
print("-" * 80)
seed_cols = [col for col in ['winner_seed', 'loser_seed'] if col in df_all.columns]
if seed_cols:
    for col in seed_cols:
        nan_count = df_all[col].isna().sum()
        print(f"  {col}: {nan_count:,} NaN kept (unseeded player)")

# 7. Indoor
print("\n7. INDOOR")
print("-" * 80)
if 'indoor' in df_all.columns:
    nan_count = df_all['indoor'].isna().sum()
    print(f"  indoor: {nan_count:,} NaN kept (unknown)")

# 8. Minutes
print("\n8. MINUTES")
print("-" * 80)
if 'minutes' in df_all.columns:
    nan_count = df_all['minutes'].isna().sum()
    print(f"  minutes: {nan_count:,} NaN kept (not critical)")

# 9. Match statistics
print("\n9. MATCH STATISTICS (w_ace, l_df, etc.)")
print("-" * 80)
stats_cols = [col for col in df_all.columns if col.startswith(('w_', 'l_'))]
stats_with_nan = [(col, df_all[col].isna().sum()) for col in stats_cols if df_all[col].isna().sum() > 0]

if stats_with_nan:
    print(f"  Found {len(stats_with_nan)} statistics columns with missing values")
    print(f"  Keeping NaN - will use for rolling features")
    print(f"  Example: {stats_with_nan[0][0]} has {stats_with_nan[0][1]:,} NaN")

# Summary
print("\n" + "=" * 80)
print("SUMMARY")
print("=" * 80)
print(f"Final size: {len(df_all):,} matches")
print(f"Lost: {initial_size - len(df_all):,} matches ({(initial_size - len(df_all)) / initial_size * 100:.2f}%)")
print(f"Columns: {len(df_all.columns)}")

critical_cols = ['winner_rank', 'loser_rank', 'draw_size']
critical_missing = sum(df_all[col].isna().sum() for col in critical_cols if col in df_all.columns)
print(f"NaN in critical columns (rank, draw_size): {critical_missing:,}")
print("=" * 80)

🔧 ОБРОБКА ПРОПУСКІВ
Початковий розмір: 39,543 матчів

1️⃣  МЕТАДАНІ ГРАВЦІВ
--------------------------------------------------------------------------------
  ✓ winner_hand: заповнено 220 пропусків → 'R'
  ✓ loser_hand: заповнено 562 пропусків → 'R'
  ✓ winner_ht: заповнено 415 пропусків → 188 см
  ✓ loser_ht: заповнено 882 пропусків → 185 см
  ✓ winner_age: заповнено 12 пропусків → 27.0 років
  ✓ loser_age: заповнено 23 пропусків → 27.2 років
  ✓ ioc: видалено 8 рядків з пропусками

2️⃣  ENTRY TYPE (тип входу в турнір)
--------------------------------------------------------------------------------
  ✓ winner_entry: заповнено 34,378 пропусків → 'DA' (Direct Acceptance)
  ✓ loser_entry: заповнено 31,395 пропусків → 'DA' (Direct Acceptance)

3️⃣  RANK та RANK_POINTS
--------------------------------------------------------------------------------
  ✓ winner_rank: заповнено 266 → 1000 (слабкий гравець)
  ✓ loser_rank: заповнено 665 → 1000
  ✓ winner_rank_points: заповнено 266 → 0
  ✓ lose

## Step 5: Train/Test Split

- **Train**: 2012-2024 (13 years)
- **Test**: 2025 (current year)

In [ ]:
df_train = df_all[df_all['data_year'] < 2025].copy()
df_test = df_all[df_all['data_year'] == 2025].copy()

print("DATA SPLIT")
print("=" * 80)
print(f"Train (2012-2024): {len(df_train):,} matches")
print(f"Test (2025):       {len(df_test):,} matches")
print(f"Ratio:             {len(df_train) / len(df_test):.1f}:1")
print("=" * 80)

print("\nTrain distribution by year:")
train_years = df_train['data_year'].value_counts().sort_index()

📊 РОЗДІЛЕННЯ ДАНИХ
Train (2012-2024): 36,624 матчів
Test (2025):       2,911 матчів
Співвідношення:    12.6:1

📅 Розподіл Train по роках:
  2012: 3,008 матчів
  2013: 2,944 матчів
  2014: 2,901 матчів
  2015: 2,945 матчів
  2016: 2,970 матчів
  2017: 2,936 матчів
  2018: 2,926 матчів
  2019: 2,806 матчів
  2020: 1,466 матчів
  2021: 2,735 матчів
  2022: 2,918 матчів
  2023: 2,995 матчів
  2024: 3,074 матчів

📅 Test (2025): 2,911 матчів


## Step 6: Saving Processed Data

In [ ]:
processed_dir = Path('../data/processed')
processed_dir.mkdir(exist_ok=True, parents=True)

train_path = processed_dir / 'train_2012_2024.csv'
test_path = processed_dir / 'test_2025.csv'

df_train.to_csv(train_path, index=False)
df_test.to_csv(test_path, index=False)

print("SAVING DATA")
print("=" * 80)
print(f"Train saved: {train_path}")
print(f"  File size: {train_path.stat().st_size / (1024**2):.2f} MB")
print(f"  Matches: {len(df_train):,}")
print()
print(f"Test saved: {test_path}")
print(f"  File size: {test_path.stat().st_size / (1024**2):.2f} MB")
print(f"  Matches: {len(df_test):,}")
print("=" * 80)
print("\nETL pipeline complete!")

💾 ЗБЕРЕЖЕННЯ ДАНИХ
✓ Train збережено: ../data/processed/train_2012_2024.csv
  Розмір файлу: 9.36 MB
  Матчів: 36,624

✓ Test збережено: ../data/processed/test_2025.csv
  Розмір файлу: 0.75 MB
  Матчів: 2,911

✅ ETL pipeline завершено!


## Missing Values Analysis: All OK

**What we see:**
- 21 columns with NaN out of 50 total
- **98,142 NaN** in dataset

**Why this is NORMAL:**

### Seed (57-76% NaN) - this is a FEATURE, not a problem
- NaN = "player is unseeded" (important information)
- Will create `is_winner_seeded`, `is_loser_seeded` (0/1)

### Statistics w_ace, l_df (5.94% NaN) - for Rolling Features
- Will use to calculate averages from past matches
- NaN will be filled when creating rolling aggregates

### Minutes (7.53% NaN) - not critical
- Match duration doesn't significantly affect result
- Can keep NaN or fill with median

**Critical columns (player metadata, rank, draw_size) - 100% NO MISSING VALUES**

In [ ]:
print("CRITICAL COLUMNS CHECK")
print("=" * 80)

critical_metadata = [
    'winner_name', 'loser_name',
    'winner_hand', 'loser_hand', 
    'winner_ht', 'loser_ht',
    'winner_age', 'loser_age',
    'winner_ioc', 'loser_ioc',
    'winner_rank', 'loser_rank',
    'winner_rank_points', 'loser_rank_points',
    'surface', 'tourney_level', 'draw_size'
]

print("\nCritical columns (player metadata, rank, tournament):")
print("-" * 80)

all_good = True
for col in critical_metadata:
    if col in df_all.columns:
        nan_count = df_all[col].isna().sum()
        status = "OK" if nan_count == 0 else "ERROR"
        print(f"  {status:6} {col:<25} NaN: {nan_count:>6,}")
        if nan_count > 0:
            all_good = False
    else:
        print(f"  WARN   {col:<25} column not present")

print("\n" + "=" * 80)
if all_good:
    print("EXCELLENT! All critical columns are filled!")
else:
    print("WARNING: Missing values in critical columns - need additional processing")

print("\nColumns with NaN (this is NORMAL for the model):")
print("-" * 80)

ok_with_nan = {
    'winner_seed': 'NaN = player is unseeded',
    'loser_seed': 'NaN = player is unseeded',
    'minutes': 'NaN = data not collected (not critical)',
}

for col, explanation in ok_with_nan.items():
    if col in df_all.columns:
        nan_count = df_all[col].isna().sum()
        pct = nan_count / len(df_all) * 100
        print(f"  OK  {col:<20} {nan_count:>6,} ({pct:>5.1f}%) - {explanation}")

stats_cols = [col for col in df_all.columns if col.startswith(('w_', 'l_'))]
stats_nan = sum(df_all[col].isna().sum() for col in stats_cols)
print(f"\n  OK  Match statistics:    {stats_nan:>6,} NaN - will use for rolling features")

print("=" * 80)
print(f"\nREADY TO SAVE:")
print(f"  Rows: {len(df_all):,}")
print(f"  Columns: {len(df_all.columns)}")
print(f"  Total NaN: {df_all.isna().sum().sum():,} (this is OK!)")
print("=" * 80)

✅ ПЕРЕВІРКА КРИТИЧНИХ КОЛОНОК

🔍 Критичні колонки (метадані гравців, rank, турнір):
--------------------------------------------------------------------------------
  ✅ winner_name               NaN:      0
  ✅ loser_name                NaN:      0
  ✅ winner_hand               NaN:      0
  ✅ loser_hand                NaN:      0
  ✅ winner_ht                 NaN:      0
  ✅ loser_ht                  NaN:      0
  ✅ winner_age                NaN:      0
  ✅ loser_age                 NaN:      0
  ✅ winner_ioc                NaN:      0
  ✅ loser_ioc                 NaN:      0
  ✅ winner_rank               NaN:      0
  ✅ loser_rank                NaN:      0
  ✅ winner_rank_points        NaN:      0
  ✅ loser_rank_points         NaN:      0
  ❌ surface                   NaN:    143
  ✅ tourney_level             NaN:      0
  ✅ draw_size                 NaN:      0

⚠️  Є пропуски в критичних колонках - потрібна додаткова обробка

📊 Колонки з NaN (це НОРМАЛЬНО для моделі):
-----------

In [ ]:
print("COMPLETE MISSING VALUES ANALYSIS AFTER PROCESSING")
print("=" * 100)

results = []

for col in df_all.columns:
    nan_count = df_all[col].isna().sum()
    empty_count = (df_all[col] == '').sum() if df_all[col].dtype == 'object' else 0
    
    if df_all[col].dtype == 'object':
        whitespace_count = (df_all[col].astype(str).str.strip() == '').sum() - nan_count
    else:
        whitespace_count = 0
    
    none_count = (df_all[col] == 'None').sum() if df_all[col].dtype == 'object' else 0
    na_count = (df_all[col] == 'NA').sum() if df_all[col].dtype == 'object' else 0
    
    total_missing = nan_count + empty_count + whitespace_count + none_count + na_count
    
    if total_missing > 0:
        results.append({
            'Column': col,
            'NaN': nan_count,
            'Empty_String': empty_count,
            'Whitespace': whitespace_count,
            'None_String': none_count,
            'NA_String': na_count,
            'Total_Missing': total_missing,
            'Percent_Missing': round(total_missing / len(df_all) * 100, 2)
        })

missing_analysis = pd.DataFrame(results).set_index('Column').sort_values(
    'Percent_Missing', ascending=False
)

print(f"\nFound {len(missing_analysis)} columns with missing values out of {len(df_all.columns)} total\n")
print("TOP 30 COLUMNS WITH MISSING VALUES:")
print("-" * 100)
print(f"{'Column':<25} {'NaN':>8} {'Empty':>8} {'Space':>8} {'None':>8} {'NA':>8} {'TOTAL':>10} {'%':>8}")
print("-" * 100)

for idx, (col, row) in enumerate(missing_analysis.head(30).iterrows(), 1):
    print(f"{col:<25} {int(row['NaN']):>8,} {int(row['Empty_String']):>8,} "
          f"{int(row['Whitespace']):>8,} {int(row['None_String']):>8,} "
          f"{int(row['NA_String']):>8,} {int(row['Total_Missing']):>10,} {row['Percent_Missing']:>7.2f}%")

🔍 ПОВНИЙ АНАЛІЗ ПРОПУСКІВ ПІСЛЯ ОБРОБКИ

📊 Знайдено 21 колонок з пропусками з 50 загальних

ТОП-30 КОЛОНОК З ПРОПУСКАМИ:
----------------------------------------------------------------------------------------------------
Колонка                        NaN    Empty    Space     None       NA     ВСЬОГО        %
----------------------------------------------------------------------------------------------------
loser_seed                  30,025        0        0        0        0     30,025   75.95%
winner_seed                 22,839        0        0        0        0     22,839   57.77%
minutes                      2,978        0        0        0        0      2,978    7.53%
l_ace                        2,350        0        0        0        0      2,350    5.94%
l_bpSaved                    2,350        0        0        0        0      2,350    5.94%
l_SvGms                      2,350        0        0        0        0      2,350    5.94%
l_2ndWon                     2,350      

In [99]:
df_all.head(5)

,tourney_id,tourney_name,surface,draw_size,tourney_level,indoor,tourney_date,winner_id,winner_seed,winner_entry,...,l_ace,l_df,l_svpt,l_1stIn,l_1stWon,l_2ndWon,l_SvGms,l_bpSaved,l_bpFaced,data_year
0,2012-339,Brisbane,Hard,32.0,250,O,20120101,MC10,1.0,DA,...,4.0,3.0,77.0,41.0,29.0,15.0,14.0,4.0,9.0,2012
1,2012-339,Brisbane,Hard,32.0,250,O,20120101,MA30,NaN,DA,...,3.0,5.0,103.0,67.0,54.0,20.0,16.0,1.0,2.0,2012
2,2012-339,Brisbane,Hard,32.0,250,O,20120101,B837,NaN,DA,...,14.0,1.0,79.0,48.0,36.0,17.0,11.0,2.0,4.0,2012
3,2012-339,Brisbane,Hard,32.0,250,O,20120101,N552,5.0,DA,...,4.0,2.0,90.0,49.0,27.0,23.0,13.0,9.0,14.0,2012
4,2012-339,Brisbane,Hard,32.0,250,O,20120101,I165,NaN,DA,...,2.0,1.0,55.0,38.0,28.0,11.0,9.0,0.0,0.0,2012
